# Latium Causal Tracing — Standalone Reference

This is the canonical, self-contained notebook for Latium's early-site causal tracing method. It imports no Latium code and can run from a fresh Jupyter or Colab environment after the optional install cell.

The method keeps the strongest parts of the Latium v2 design: architecture-specific final-MLP-projection hooks, corruption/restoration paired by noise sample, restoration at the last subject token over equal-width layer windows, and a discovery/held-out-confirmation decision that never consults the configured ROME layer.

The notebook and `python -m src causal-trace` implement the same causal estimand and selector. The notebook is the inspectable reference; the package implementation is the scalable CLI.


## 0. Optional Install Cell

In [ ]:
# Uncomment in Colab or a fresh environment.
# %pip install -q torch transformers datasets accelerate pandas matplotlib tqdm
# Optional for quantized loading on limited VRAM:
# %pip install -q bitsandbytes

## 1. Imports

In [ ]:
from __future__ import annotations

from collections import Counter
from contextlib import contextmanager
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
import gc
import json
import math
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import DatasetDict, concatenate_datasets, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.grid'] = True
print('Imports OK')

## 2. Batch Settings

In [ ]:
# Run one or many model keys from MODEL_PRESETS below. Start with one model.
MODEL_CONFIGS = ['gpt2-xl']

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'bf16'  # bf16, f16, f32, auto
USE_DEVICE_MAP_AUTO = False
TRUST_REMOTE_CODE = True
LOAD_IN_4BIT = False
ADAPTER_VALIDATE_ALL_LAYERS = True

DATASET_NAME = 'azhx/counterfact'
DATASET_SPLITS = ['train', 'test']
NUM_VALID_FACTS = 100
DISCOVERY_FRACTION = 0.5
MAX_DATASET_EXAMPLES_TO_SCAN = 10000
TRACE_STATUS_EVERY = 25
NUM_NOISE_SAMPLES = 10
NOISE_BATCH_SIZE = 2
NOISE_MULTIPLIER = 3.0
SEED = 42

WINDOW_MODE = 'canonical_rome'  # canonical_rome or proportional
WINDOW_SIZE = 10
WINDOW_FRACTION = 0.20

REQUIRE_CORRECT_CLEAN = True
MIN_TOTAL_EFFECT = 0.03

# Technical floor only. Production runs should use substantially more held-out facts.
MIN_CONFIRMATION_FACTS = 2
BOOTSTRAP_SAMPLES = 1000
CONFIDENCE_LEVEL = 0.95

# Graph-only references copied from Latium model configs. They never affect selection.
CONFIG_REFERENCE_LAYERS = {
    'gpt2-medium': 8, 'gpt2-large': 12, 'gpt2-xl': 18, 'gpt-j-6b': 5,
    'falcon-7b': 3, 'opt-6.7b': 15, 'llama2-7b': 19,
    'mistral-7b-v0.1': 5, 'mistral-7b-v0.3': 17,
    'deepseek-7b-base': 6, 'deepseek-r1-llama3-8b': 0,
    'qwen2.5-1.5b': 7, 'qwen3-0.6b': 5, 'qwen3-1.7b': 9,
    'qwen3-4b': 12, 'qwen3-8b': 10, 'granite4-micro': 9,
    'qwen3-guard-0.6b': 5,
}

SAVE = True
OUT_ROOT = Path('./analysis_out/causal_tracing')
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_CAUSAL_TRACE = True

if not 0 < DISCOVERY_FRACTION < 1:
    raise ValueError('DISCOVERY_FRACTION must be strictly between 0 and 1')
if MIN_CONFIRMATION_FACTS < 2:
    raise ValueError('MIN_CONFIRMATION_FACTS must be at least 2')
if NUM_NOISE_SAMPLES <= 0 or NOISE_BATCH_SIZE <= 0 or BOOTSTRAP_SAMPLES <= 0:
    raise ValueError('Noise sample, noise batch, and bootstrap counts must be positive')
if not 0 < CONFIDENCE_LEVEL < 1:
    raise ValueError('CONFIDENCE_LEVEL must be strictly between 0 and 1')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## 3. Built-In Model Presets

In [ ]:
MODEL_PRESETS = {
    'gpt2-medium': {'name': 'gpt2-medium', 'layers': 24, 'embedding': 'transformer.wte', 'block_template': 'transformer.h.{}', 'mlp_template': 'transformer.h.{}.mlp.c_proj', 'dtype': 'bf16'},
    'gpt2-large': {'name': 'gpt2-large', 'layers': 36, 'embedding': 'transformer.wte', 'block_template': 'transformer.h.{}', 'mlp_template': 'transformer.h.{}.mlp.c_proj', 'dtype': 'bf16'},
    'gpt2-xl': {'name': 'gpt2-xl', 'layers': 48, 'embedding': 'transformer.wte', 'block_template': 'transformer.h.{}', 'mlp_template': 'transformer.h.{}.mlp.c_proj', 'dtype': 'bf16'},
    'gpt-j-6b': {'name': 'EleutherAI/gpt-j-6B', 'layers': 28, 'embedding': 'transformer.wte', 'block_template': 'transformer.h.{}', 'mlp_template': 'transformer.h.{}.mlp.fc_out', 'dtype': 'bf16'},
    'falcon-7b': {'name': 'tiiuae/falcon-7b', 'layers': 32, 'embedding': 'transformer.word_embeddings', 'block_template': 'transformer.h.{}', 'mlp_template': 'transformer.h.{}.mlp.dense_4h_to_h', 'dtype': 'bf16', 'trust_remote_code': False},
    'opt-6.7b': {'name': 'facebook/opt-6.7b', 'layers': 32, 'embedding': 'model.decoder.embed_tokens', 'block_template': 'model.decoder.layers.{}', 'mlp_template': 'model.decoder.layers.{}.fc2', 'dtype': 'bf16'},
    'llama2-7b': {'name': 'NousResearch/Llama-2-7b-hf', 'layers': 32, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'mistral-7b-v0.1': {'name': 'mistralai/Mistral-7B-v0.1', 'layers': 32, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'mistral-7b-v0.3': {'name': 'mistralai/Mistral-7B-v0.3', 'layers': 32, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'deepseek-7b-base': {'name': 'deepseek-ai/deepseek-llm-7b-base', 'layers': 30, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'deepseek-r1-llama3-8b': {'name': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B', 'layers': 32, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'qwen2.5-1.5b': {'name': 'Qwen/Qwen2.5-Math-1.5B', 'layers': 28, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'qwen3-0.6b': {'name': 'Qwen/Qwen3-0.6B', 'layers': 28, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'qwen3-1.7b': {'name': 'Qwen/Qwen3-1.7B', 'layers': 28, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'qwen3-4b': {'name': 'Qwen/Qwen3-4B', 'layers': 36, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'qwen3-8b': {'name': 'Qwen/Qwen3-8B', 'layers': 36, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
    'granite4-micro': {'name': 'ibm-granite/granite-4.0-micro', 'layers': 40, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.shared_mlp.output_linear', 'dtype': 'bf16'},
    'qwen3-guard-0.6b': {'name': 'Qwen/Qwen3Guard-Gen-0.6B', 'layers': 28, 'embedding': 'model.embed_tokens', 'block_template': 'model.layers.{}', 'mlp_template': 'model.layers.{}.mlp.down_proj', 'dtype': 'bf16'},
}
unknown = [name for name in MODEL_CONFIGS if name not in MODEL_PRESETS]
if unknown:
    raise ValueError(f'Unknown MODEL_CONFIGS={unknown}. Available: {sorted(MODEL_PRESETS)}')
print('Batch models:', MODEL_CONFIGS)

## 4. Data Types

In [ ]:
@dataclass
class WindowTrace:
    center: int
    start: int
    end: int
    layers: list[int]
    module_names: list[str]
    is_full_width: bool
    restore_probabilities: np.ndarray
    ie_samples: np.ndarray
    mean_ie: float
    median_ie: float
    std_ie: float
    sem_ie: float
    normalized_recovery: float

@dataclass
class FactTrace:
    prompt_idx: int
    prompt: str
    subject: str
    target_full_text: str
    target_first_token_id: int
    target_first_token_text: str
    target_num_tokens: int
    clean_top_token: str
    clean_probability: float
    clean_top_probability: float
    corrupt_probabilities: np.ndarray
    total_effect: float
    corrupt_relative_std: float
    subject_positions: list[int]
    subject_tokens: list[str]
    subject_last_position: int
    subject_last_token: str
    prompt_last_position: int
    prompt_last_token: str
    windows: list[WindowTrace]

    @property
    def mean_corrupt_probability(self):
        return float(np.mean(self.corrupt_probabilities))

class TraceSkip(Exception):
    def __init__(self, reason, detail):
        super().__init__(detail)
        self.reason = reason
        self.detail = detail

## 5. Shared Helpers

In [ ]:
def normalize_counterfact_dataset(raw):
    parts = []
    if isinstance(raw, DatasetDict):
        for split in DATASET_SPLITS:
            if split in raw:
                parts.append(raw[split])
        if not parts:
            parts = [next(iter(raw.values()))]
        return concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    return raw

@contextmanager
def temporary_hooks(hooks):
    handles = []
    try:
        for module, hook in hooks:
            handles.append(module.register_forward_hook(hook))
        yield
    finally:
        for handle in handles:
            handle.remove()

def hidden_from_output(output):
    return output[0] if isinstance(output, tuple) else output

def replace_hidden(output, hidden):
    if isinstance(output, tuple):
        values = list(output)
        values[0] = hidden
        return tuple(values)
    return hidden

def repeat_inputs(inputs, repeats: int):
    return {key: value.repeat((repeats,) + (1,) * (value.dim() - 1)) for key, value in inputs.items() if torch.is_tensor(value)}

def bootstrap_ci(matrix, samples=BOOTSTRAP_SAMPLES, confidence=CONFIDENCE_LEVEL, seed=SEED):
    matrix = np.asarray(matrix)
    if matrix.shape[0] == 1:
        return matrix[0].copy(), matrix[0].copy()
    rng = np.random.default_rng(int(seed))
    boot = np.empty((int(samples), matrix.shape[1]), dtype=np.float64)
    n = matrix.shape[0]
    for sample_idx in range(int(samples)):
        idx = rng.integers(0, n, size=n)
        boot[sample_idx] = matrix[idx].mean(axis=0)
    alpha = (1.0 - float(confidence)) / 2.0
    return np.quantile(boot, alpha, axis=0), np.quantile(boot, 1.0 - alpha, axis=0)

def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    return value

## 6. Load Dataset Once

In [ ]:
print(f'Loading dataset {DATASET_NAME}...')
raw_dataset = load_dataset(DATASET_NAME)
dataset = normalize_counterfact_dataset(raw_dataset)
print(dataset)

## 7. Batch Runner

In [ ]:
def run_one_model(model_config: str, dataset):
    preset = MODEL_PRESETS[model_config]
    out_dir = OUT_ROOT / f'{model_config}_{RUN_TIMESTAMP}'
    dtype_picker = {'auto': 'auto', 'bf16': torch.bfloat16, 'f16': torch.float16, 'f32': torch.float32}
    dtype = dtype_picker.get(DTYPE, dtype_picker.get(preset.get('dtype', 'auto'), 'auto'))
    model_name = preset['name']
    model_trust_remote_code = bool(preset.get('trust_remote_code', TRUST_REMOTE_CODE))

    def ensure_padding(tok):
        if tok.pad_token is None:
            if tok.eos_token is not None:
                tok.pad_token = tok.eos_token
            elif tok.eos_token_id is not None:
                tok.pad_token_id = tok.eos_token_id
        if tok.pad_token_id is None and tok.eos_token_id is not None:
            tok.pad_token_id = tok.eos_token_id
        return tok

    model_kwargs = {'trust_remote_code': model_trust_remote_code}
    if dtype != 'auto':
        model_kwargs['torch_dtype'] = dtype
    if os.environ.get('HF_TOKEN'):
        model_kwargs['token'] = os.environ['HF_TOKEN']
    if USE_DEVICE_MAP_AUTO:
        model_kwargs['device_map'] = 'auto'
    if LOAD_IN_4BIT:
        model_kwargs['load_in_4bit'] = True
        model_kwargs['device_map'] = 'auto'

    print(f'\n===== Loading {model_config}: {model_name} (trust_remote_code={model_trust_remote_code}) =====')
    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    if not USE_DEVICE_MAP_AUTO and not LOAD_IN_4BIT:
        model = model.to(DEVICE)
    model.eval()
    tok_kwargs = {'trust_remote_code': model_trust_remote_code}
    if os.environ.get('HF_TOKEN'):
        tok_kwargs['token'] = os.environ['HF_TOKEN']
    tokenizer = ensure_padding(AutoTokenizer.from_pretrained(model_name, **tok_kwargs))
    num_layers = int(getattr(model.config, 'num_hidden_layers', preset.get('layers')))
    input_device = next(model.parameters()).device
    primary_window_size = max(1, int(round(num_layers * WINDOW_FRACTION))) if WINDOW_MODE == 'proportional' else int(WINDOW_SIZE)
    if not 1 <= primary_window_size <= num_layers:
        raise ValueError(f'Window size must be between 1 and {num_layers}, got {primary_window_size}')
    config_reference_layer = CONFIG_REFERENCE_LAYERS.get(model_config)
    print(f'input_device={input_device}; num_layers={num_layers}; window_size={primary_window_size}; config_reference_layer={config_reference_layer}')

    available_module_names = {name for name, _module in model.named_modules()}

    def resolve_module(name: str):
        for module_name, module in model.named_modules():
            if module_name == name:
                return module
        raise KeyError(f'Module not found: {name}')

    def embedding_module_name():
        name = preset['embedding']
        if name not in available_module_names:
            raise KeyError(f'Embedding module {name!r} not found')
        return name

    def candidate_mlp_output_module_names(layer: int):
        primary = preset['mlp_template'].format(int(layer))
        candidates = [primary]
        for marker in ('.mlp.', '.shared_mlp.'):
            if marker in primary:
                candidates.append(primary.split(marker, 1)[0] + marker.rstrip('.'))
        block = preset.get('block_template', '').format(int(layer)) if preset.get('block_template') else ''
        if block:
            candidates.extend([f'{block}.mlp', f'{block}.shared_mlp', f'{block}.feed_forward', f'{block}.ffn', f'{block}.fc2', f'{block}.mlp.dense_4h_to_h'])
        candidates.extend([f'transformer.h.{int(layer)}.mlp', f'model.layers.{int(layer)}.mlp', f'model.layers.{int(layer)}.shared_mlp', f'model.decoder.layers.{int(layer)}.fc2'])
        unique = []
        for name in candidates:
            if name and name not in unique:
                unique.append(name)
        return unique

    def mlp_output_module_name(layer: int):
        for name in candidate_mlp_output_module_names(layer):
            if name in available_module_names:
                return name
        raise KeyError(f'Could not resolve MLP-output module for layer {layer}. Tried {candidate_mlp_output_module_names(layer)[:8]}')

    module_map = pd.DataFrame([{'layer': layer, 'mlp_output_module': mlp_output_module_name(layer)} for layer in range(num_layers)])

    def tokenize_prompt(prompt_text: str):
        return tokenizer(prompt_text, return_tensors='pt', padding=False).to(input_device)

    def window_layers(center: int):
        left_width = primary_window_size // 2
        right_width = primary_window_size - left_width
        start = max(0, int(center) - left_width)
        end = min(int(num_layers), int(center) + right_width)
        return list(range(start, end))

    def window_metadata(center: int):
        layers = window_layers(center)
        return {'window_center': int(center), 'window_start': int(layers[0]), 'window_end': int(layers[-1] + 1), 'window_size_actual': int(len(layers)), 'window_layers': layers, 'window_is_full_width': len(layers) == primary_window_size, 'excluded_from_ranking': len(layers) != primary_window_size, 'exclusion_reason': None if len(layers) == primary_window_size else 'partial_boundary_window'}

    def target_token_info(target: str):
        cleaned = str(target).strip()
        candidates = [
            (f' {cleaned}', False), (cleaned, False),
            (f' {cleaned}', True), (cleaned, True),
        ]
        for text, add_special_tokens in candidates:
            ids = tokenizer(text, add_special_tokens=add_special_tokens)['input_ids']
            if ids and isinstance(ids[0], list):
                ids = ids[0]
            bos = getattr(tokenizer, 'bos_token_id', None)
            if bos is not None and len(ids) > 1 and ids[0] == bos:
                ids = ids[1:]
            if ids:
                return int(ids[0]), tokenizer.decode([int(ids[0])]), len(ids)
        raise ValueError(f'Could not tokenize target {target!r}')

    def subject_span_from_offsets(prompt: str, subject: str):
        starts = []
        cursor = 0
        while True:
            idx = prompt.find(subject, cursor)
            if idx == -1:
                break
            starts.append(idx)
            cursor = idx + max(1, len(subject))
        if len(starts) != 1:
            raise ValueError(f'Subject span ambiguous or missing: found {len(starts)} matches')
        char_start = starts[0]
        char_end = char_start + len(subject)
        try:
            encoded = tokenizer(prompt, return_offsets_mapping=True, return_tensors='pt')
            offsets = encoded.get('offset_mapping')
        except Exception:
            offsets = None
        if offsets is not None:
            positions = [
                int(idx)
                for idx, (start, end) in enumerate(offsets[0].detach().cpu().tolist())
                if end > start and end > char_start and start < char_end
            ]
            if positions:
                return positions

        prompt_ids = tokenizer(prompt)['input_ids']
        candidates = []
        for text, add_special_tokens in [
            (subject, False), (f' {subject}', False),
            (subject, True), (f' {subject}', True),
        ]:
            ids = tokenizer(text, add_special_tokens=add_special_tokens)['input_ids']
            bos = getattr(tokenizer, 'bos_token_id', None)
            if bos is not None and len(ids) > 1 and ids[0] == bos:
                ids = ids[1:]
            if ids:
                candidates.append(ids)
        matches = set()
        for ids in candidates:
            for start in range(len(prompt_ids) - len(ids) + 1):
                if prompt_ids[start:start + len(ids)] == ids:
                    matches.add((start, start + len(ids)))
        if len(matches) != 1:
            raise ValueError(f'Could not identify a unique subject token span for {subject!r}')
        start, end = next(iter(matches))
        return list(range(start, end))

    def decode_position(input_ids, position: int):
        return tokenizer.decode([int(input_ids[0, int(position)].detach().cpu().item())])

    def make_noise(num_samples, subject_len, hidden_size, noise_std, device, model_dtype, seed):
        gen = torch.Generator(device='cpu')
        gen.manual_seed(int(seed))
        noise = torch.randn((num_samples, subject_len, hidden_size), generator=gen, dtype=torch.float32)
        return (noise * float(noise_std)).to(device=device, dtype=model_dtype)

    def corrupt_hook(subject_positions, noise_samples):
        positions = [int(pos) for pos in subject_positions]
        def hook(_module, _inputs, output):
            hidden = hidden_from_output(output)
            changed = hidden.clone()
            noise = noise_samples.to(device=changed.device, dtype=changed.dtype)
            for offset, token_idx in enumerate(positions):
                changed[:, token_idx, :] = changed[:, token_idx, :] + noise[:, offset, :]
            return replace_hidden(output, changed)
        return hook

    def mlp_state_at_position(hidden: torch.Tensor, position: int, sequence_length: int):
        position = int(position)
        sequence_length = int(sequence_length)
        if hidden.dim() == 3:
            if hidden.shape[1] <= position:
                raise RuntimeError(f'MLP output sequence length {hidden.shape[1]} does not contain position {position}')
            return hidden[0, position, :].detach().clone()
        if hidden.dim() == 2:
            if hidden.shape[0] % sequence_length != 0:
                raise RuntimeError(f'2D MLP output first dimension {hidden.shape[0]} is not divisible by sequence length {sequence_length}')
            row = position
            if row >= hidden.shape[0]:
                raise RuntimeError(f'2D MLP output shape {tuple(hidden.shape)} does not contain position {position}')
            return hidden[row, :].detach().clone()
        raise RuntimeError(f'Unsupported MLP output rank {hidden.dim()} with shape {tuple(hidden.shape)}')

    def patch_mlp_position(hidden: torch.Tensor, position: int, clean_state: torch.Tensor, sequence_length: int):
        position = int(position)
        sequence_length = int(sequence_length)
        changed = hidden.clone()
        state = clean_state.to(device=changed.device, dtype=changed.dtype)
        if hidden.dim() == 3:
            changed[:, position, :] = state
            return changed
        if hidden.dim() == 2:
            if hidden.shape[0] % sequence_length != 0:
                raise RuntimeError(f'2D MLP output first dimension {hidden.shape[0]} is not divisible by sequence length {sequence_length}')
            batch_count = hidden.shape[0] // sequence_length
            rows = torch.arange(batch_count, device=changed.device, dtype=torch.long) * sequence_length + position
            changed[rows, :] = state
            return changed
        raise RuntimeError(f'Unsupported MLP output rank {hidden.dim()} with shape {tuple(hidden.shape)}')

    def supported_mlp_output_shape(shape, sequence_length: int):
        if shape is None:
            return False
        if len(shape) == 3:
            return shape[0] == 1 and shape[1] == int(sequence_length)
        if len(shape) == 2:
            return shape[0] % int(sequence_length) == 0
        return False

    def restore_position_hook(position: int, clean_state: torch.Tensor, sequence_length: int):
        def hook(_module, _inputs, output):
            hidden = hidden_from_output(output)
            changed = patch_mlp_position(hidden, position, clean_state, sequence_length)
            return replace_hidden(output, changed)
        return hook

    def token_probability(logits, token_id: int):
        return torch.softmax(logits[:, -1, :], dim=-1)[:, int(token_id)].detach().float().cpu().numpy()

    def mlp_shape_mode(shape, sequence_length: int):
        if shape is None:
            return 'missing'
        if len(shape) == 3 and shape[0] == 1 and shape[1] == int(sequence_length):
            return 'batch_seq_hidden'
        if len(shape) == 2 and shape[0] % int(sequence_length) == 0:
            return 'flat_batch_seq_hidden'
        return 'unsupported'

    def validate_mlp_adapter_shapes():
        # Adapter validation uses a neutral prompt and never reads configured/reference layers for selection.
        probe_inputs = tokenize_prompt('The capital of France is')
        probe_sequence_length = int(probe_inputs['input_ids'].shape[1])
        layers_to_probe = range(num_layers) if ADAPTER_VALIDATE_ALL_LAYERS else [num_layers // 2]
        captured = {}
        hooks = []
        for layer in layers_to_probe:
            name = mlp_output_module_name(int(layer))
            def make_hook(layer_idx, module_name):
                def hook(_module, _inputs, output):
                    hidden = hidden_from_output(output)
                    captured[int(layer_idx)] = {'shape': tuple(hidden.shape), 'module_name': module_name}
                    return output
                return hook
            hooks.append((resolve_module(name), make_hook(int(layer), name)))
        with torch.inference_mode(), temporary_hooks(hooks):
            model(**probe_inputs, use_cache=False)
        rows = []
        for layer in layers_to_probe:
            layer = int(layer)
            item = captured.get(layer, {'shape': None, 'module_name': mlp_output_module_name(layer)})
            shape = item['shape']
            mode = mlp_shape_mode(shape, probe_sequence_length)
            if mode in {'missing', 'unsupported'}:
                raise RuntimeError(
                    f'Adapter validation failed for {model_config} layer {layer} module {item["module_name"]}: '
                    f'captured MLP output shape {shape}; expected 3D [batch, seq, hidden] or 2D [batch*seq, hidden]'
                )
            rows.append({'layer': layer, 'mlp_output_module': item['module_name'], 'probe_shape': shape, 'shape_mode': mode})
        df = pd.DataFrame(rows)
        modes = ', '.join(f'{mode}:{count}' for mode, count in df['shape_mode'].value_counts().sort_index().items())
        print(f'Adapter validation OK: {len(df)} layer(s), shape modes: {modes}')
        return df

    adapter_validation = validate_mlp_adapter_shapes()
    module_map = module_map.merge(adapter_validation, on=['layer', 'mlp_output_module'], how='left')

    def cache_clean_mlp_outputs(inputs, position: int):
        captured = {}
        hooks = []
        chosen_names = []
        sequence_length = int(inputs['input_ids'].shape[1])
        for layer in range(num_layers):
            name = mlp_output_module_name(layer)
            chosen_names.append(name)
            def make_hook(layer_idx):
                def hook(_module, _inputs, output):
                    hidden = hidden_from_output(output)
                    captured[layer_idx] = mlp_state_at_position(hidden, position, sequence_length)
                    return output
                return hook
            hooks.append((resolve_module(name), make_hook(layer)))
        with torch.inference_mode(), temporary_hooks(hooks):
            model(**inputs, use_cache=False)
        return {'position': int(position), 'sequence_length': sequence_length, 'module_names': chosen_names, 'states': {layer: captured[layer].detach().clone() for layer in captured}}

    def trace_mlp_windows_for_fact(inputs, target_id, subject_positions, noise_samples, corrupt_probabilities, clean_probability, restore_position):
        emb = resolve_module(embedding_module_name())
        clean_cache = cache_clean_mlp_outputs(inputs, restore_position)
        windows = []
        noise_batch_size = max(1, min(int(NOISE_BATCH_SIZE), int(NUM_NOISE_SAMPLES)))
        with torch.inference_mode():
            for center in range(num_layers):
                meta = window_metadata(center)
                restore_probs = np.zeros((NUM_NOISE_SAMPLES,), dtype=np.float32)
                module_names_for_window = [clean_cache['module_names'][layer] for layer in meta['window_layers']]
                for batch_start in range(0, NUM_NOISE_SAMPLES, noise_batch_size):
                    batch_end = min(NUM_NOISE_SAMPLES, batch_start + noise_batch_size)
                    repeated = repeat_inputs(inputs, batch_end - batch_start)
                    hooks = [(emb, corrupt_hook(subject_positions, noise_samples[batch_start:batch_end]))]
                    for layer in meta['window_layers']:
                        name = clean_cache['module_names'][layer]
                        hooks.append((resolve_module(name), restore_position_hook(restore_position, clean_cache['states'][layer], clean_cache['sequence_length'])))
                    with temporary_hooks(hooks):
                        restored = model(**repeated, use_cache=False)
                    restore_probs[batch_start:batch_end] = token_probability(restored.logits, target_id)
                    del repeated, restored
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                ie = restore_probs - corrupt_probabilities
                windows.append(WindowTrace(center=int(center), start=meta['window_start'], end=meta['window_end'], layers=meta['window_layers'], module_names=module_names_for_window, is_full_width=meta['window_is_full_width'], restore_probabilities=restore_probs, ie_samples=ie, mean_ie=float(np.mean(ie)), median_ie=float(np.median(ie)), std_ie=float(np.std(ie)), sem_ie=float(np.std(ie) / max(np.sqrt(len(ie)), 1.0)), normalized_recovery=float(np.mean(ie) / max(abs(clean_probability - float(np.mean(corrupt_probabilities))), 1e-9))))
        return windows

    def row_to_fact(row):
        rr = row.get('requested_rewrite', row)
        subject = rr['subject']
        target = rr['target_true']['str'] if isinstance(rr.get('target_true'), dict) else rr.get('target')
        prompt = rr['prompt'].format(subject)
        return prompt, subject, target

    def calibrate_subject_noise(dataset, max_scan):
        # Match the actual token rows corrupted by this trace. Repeated subject
        # tokens keep their multiplicity; weighted moments avoid a large cache.
        token_counts = Counter()
        rejected_subjects = 0
        accepted_subjects = 0
        for row_idx in range(int(max_scan)):
            try:
                prompt, subject, _target = row_to_fact(dataset[row_idx])
                positions = subject_span_from_offsets(prompt, subject)
                encoded = tokenizer(prompt, return_tensors='pt')
                token_counts.update(
                    int(token_id) for token_id in encoded['input_ids'][0, positions].tolist()
                )
                accepted_subjects += 1
            except Exception:
                rejected_subjects += 1
        if not token_counts:
            raise RuntimeError('No candidate subject-token embeddings available for noise calibration')

        emb = resolve_module(embedding_module_name())
        weight = emb.weight
        total = 0.0
        total_sq = 0.0
        scalar_count = 0
        items = sorted(token_counts.items())
        for start in range(0, len(items), 512):
            chunk = items[start:start + 512]
            ids = torch.tensor([token_id for token_id, _count in chunk], device=weight.device)
            counts = torch.tensor([count for _token_id, count in chunk], dtype=torch.float64)
            vectors = weight.detach().index_select(0, ids).float().cpu().to(torch.float64)
            total += float((vectors.sum(dim=1) * counts).sum().item())
            total_sq += float((vectors.square().sum(dim=1) * counts).sum().item())
            scalar_count += int(counts.sum().item()) * int(vectors.shape[1])
        variance = max(0.0, (total_sq - total * total / scalar_count) / (scalar_count - 1))
        embedding_std = math.sqrt(variance)
        return {
            'source': 'candidate_subject_token_embeddings',
            'embedding_std': embedding_std,
            'noise_multiplier': float(NOISE_MULTIPLIER),
            'noise_std': float(NOISE_MULTIPLIER) * embedding_std,
            'num_subjects': accepted_subjects,
            'num_subject_tokens': int(sum(token_counts.values())),
            'num_unique_token_ids': len(token_counts),
            'num_rejected_subjects': rejected_subjects,
        }

    def trace_fact(prompt_idx: int, prompt: str, subject: str, target: str):
        inputs = tokenize_prompt(prompt)
        subject_positions = subject_span_from_offsets(prompt, subject)
        subject_last = int(subject_positions[-1])
        prompt_last = int(inputs['input_ids'].shape[1] - 1)
        target_id, target_first_text, target_num_tokens = target_token_info(target)
        emb = resolve_module(embedding_module_name())
        noise_std = float(noise_calibration['noise_std'])
        hidden_size = int(emb.weight.shape[1])
        with torch.inference_mode():
            clean = model(**inputs, use_cache=False)
            probs = torch.softmax(clean.logits[:, -1, :], dim=-1)[0]
            clean_probability = float(probs[target_id].detach().float().cpu().item())
            clean_top_id = int(torch.argmax(probs).detach().cpu().item())
            clean_top_probability = float(probs[clean_top_id].detach().float().cpu().item())
            if REQUIRE_CORRECT_CLEAN and clean_top_id != target_id:
                raise TraceSkip('clean_mismatch', f"clean-token mismatch: top={tokenizer.decode([clean_top_id])!r} p={clean_top_probability:.6g}; target={target_first_text!r} p={clean_probability:.6g}")
            noise = make_noise(NUM_NOISE_SAMPLES, len(subject_positions), hidden_size, noise_std, inputs['input_ids'].device, next(model.parameters()).dtype, SEED + int(prompt_idx))
            noise_batch_size = max(1, min(int(NOISE_BATCH_SIZE), int(NUM_NOISE_SAMPLES)))
            corrupt_probabilities = np.zeros((NUM_NOISE_SAMPLES,), dtype=np.float32)
            for batch_start in range(0, NUM_NOISE_SAMPLES, noise_batch_size):
                batch_end = min(NUM_NOISE_SAMPLES, batch_start + noise_batch_size)
                repeated = repeat_inputs(inputs, batch_end - batch_start)
                with temporary_hooks([(emb, corrupt_hook(subject_positions, noise[batch_start:batch_end]))]):
                    corrupt = model(**repeated, use_cache=False)
                corrupt_probabilities[batch_start:batch_end] = token_probability(corrupt.logits, target_id)
                del repeated, corrupt
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        if not np.all(np.isfinite(corrupt_probabilities)) or not np.isfinite(clean_probability):
            raise TraceSkip('nonfinite_output', 'non-finite clean/corrupt probability')
        total_effect = float(clean_probability - np.mean(corrupt_probabilities))
        if total_effect < MIN_TOTAL_EFFECT:
            raise TraceSkip('low_corruption_effect', f'total_effect={total_effect:.6g} < {MIN_TOTAL_EFFECT}')
        corrupt_relative_std = float(np.std(corrupt_probabilities) / max(abs(total_effect), 1e-9))
        windows = trace_mlp_windows_for_fact(inputs, target_id, subject_positions, noise, corrupt_probabilities, clean_probability, subject_last)
        return FactTrace(prompt_idx=int(prompt_idx), prompt=prompt, subject=subject, target_full_text=target, target_first_token_id=target_id, target_first_token_text=target_first_text, target_num_tokens=target_num_tokens, clean_top_token=tokenizer.decode([clean_top_id]), clean_probability=clean_probability, clean_top_probability=clean_top_probability, corrupt_probabilities=corrupt_probabilities, total_effect=total_effect, corrupt_relative_std=corrupt_relative_std, subject_positions=subject_positions, subject_tokens=[decode_position(inputs['input_ids'], pos) for pos in subject_positions], subject_last_position=subject_last, subject_last_token=decode_position(inputs['input_ids'], subject_last), prompt_last_position=prompt_last, prompt_last_token=decode_position(inputs['input_ids'], prompt_last), windows=windows)

    # Freeze the calibration population and scale before observing trace outcomes.
    target_valid_facts = int(NUM_VALID_FACTS)
    max_scan_limit = int(MAX_DATASET_EXAMPLES_TO_SCAN)
    max_scan = min(max_scan_limit, len(dataset))
    noise_calibration = calibrate_subject_noise(dataset, max_scan)
    print(f"Noise calibration: {json.dumps(noise_calibration, indent=2)}", flush=True)
    trace_started_at = time.time()
    fact_results = []
    rejections = []
    counts = {
        'num_dataset_examples_scanned': 0,
        'num_clean_prediction_matches': 0,
        'num_baseline_reliable_facts': 0,
        'num_rejected_clean_mismatch': 0,
        'num_rejected_low_corruption_effect': 0,
        'num_rejected_nonfinite_output': 0,
        'num_rejected_other': 0,
        'target_valid_facts': target_valid_facts,
    }
    print(f'Tracing {model_config}: collecting {target_valid_facts} valid facts from up to {max_scan} scanned rows.', flush=True)
    progress = tqdm(range(max_scan), total=max_scan, desc=f'{model_config} facts')
    for prompt_idx in progress:
        if len(fact_results) >= target_valid_facts:
            break
        counts['num_dataset_examples_scanned'] += 1
        row = dataset[int(prompt_idx)]
        try:
            prompt, subject, target = row_to_fact(row)
            trace = trace_fact(prompt_idx, prompt, subject, target)
            counts['num_clean_prediction_matches'] += 1
            counts['num_baseline_reliable_facts'] += 1
            fact_results.append(trace)
            full = [w for w in trace.windows if w.is_full_width]
            best = max(full, key=lambda w: w.mean_ie)
            progress.set_postfix(valid=len(fact_results), scanned=counts['num_dataset_examples_scanned'], clean_mismatch=counts['num_rejected_clean_mismatch'])
            print(f"OK {len(fact_results):03d}/{target_valid_facts}: idx={prompt_idx} {subject!r}->{target!r} total={trace.total_effect:.4f} best_center={best.center} IE={best.mean_ie:.4g}", flush=True)
        except TraceSkip as exc:
            try:
                _prompt, subject, target = row_to_fact(row)
            except Exception:
                subject, target = '<unknown>', '<unknown>'
            if exc.reason == 'clean_mismatch':
                counts['num_rejected_clean_mismatch'] += 1
            elif exc.reason == 'low_corruption_effect':
                counts['num_clean_prediction_matches'] += 1
                counts['num_rejected_low_corruption_effect'] += 1
            elif exc.reason == 'nonfinite_output':
                counts['num_rejected_nonfinite_output'] += 1
            else:
                counts['num_rejected_other'] += 1
            rejections.append({'prompt_idx': prompt_idx, 'subject': subject, 'target': target, 'reason': exc.reason, 'detail': exc.detail})
            progress.set_postfix(valid=len(fact_results), scanned=counts['num_dataset_examples_scanned'], clean_mismatch=counts['num_rejected_clean_mismatch'])
            if TRACE_STATUS_EVERY and counts['num_dataset_examples_scanned'] % int(TRACE_STATUS_EVERY) == 0:
                print(f"TRACE {model_config}: scanned={counts['num_dataset_examples_scanned']} valid={len(fact_results)}/{target_valid_facts} clean_mismatch={counts['num_rejected_clean_mismatch']} low_effect={counts['num_rejected_low_corruption_effect']}", flush=True)
    counts['trace_elapsed_seconds'] = float(time.time() - trace_started_at)
    if len(fact_results) < 2:
        raise RuntimeError(f'{model_config}: need at least two baseline-reliable facts')

    summary = pd.DataFrame([{'prompt_idx': t.prompt_idx, 'subject': t.subject, 'target_full_text': t.target_full_text, 'target_first_token_id': t.target_first_token_id, 'target_first_token_text': repr(t.target_first_token_text), 'target_num_tokens': t.target_num_tokens, 'clean_top_token': repr(t.clean_top_token), 'clean_probability': t.clean_probability, 'mean_corrupt_probability': t.mean_corrupt_probability, 'total_effect': t.total_effect, 'corrupt_relative_std': t.corrupt_relative_std, 'subject_positions': t.subject_positions, 'subject_tokens': t.subject_tokens, 'subject_last_position': t.subject_last_position, 'subject_last_token': repr(t.subject_last_token), 'prompt_last_position': t.prompt_last_position, 'prompt_last_token': repr(t.prompt_last_token)} for t in fact_results])
    rejections_df = pd.DataFrame(rejections)

    rng = np.random.default_rng(SEED)
    indices = rng.permutation(len(fact_results))
    discovery_count = max(1, int(round(len(fact_results) * DISCOVERY_FRACTION)))
    discovery_count = min(discovery_count, len(fact_results) - 1)
    discovery_indices = sorted(indices[:discovery_count].tolist())
    confirmation_indices = sorted(indices[discovery_count:].tolist())
    discovery_traces = [fact_results[i] for i in discovery_indices]
    confirmation_traces = [fact_results[i] for i in confirmation_indices]
    split_by_index = {idx: 'discovery' for idx in discovery_indices}
    split_by_index.update({idx: 'confirmation' for idx in confirmation_indices})
    split_assignments = pd.DataFrame([
        {
            'fact_result_index': idx,
            'prompt_idx': int(trace.prompt_idx),
            'subject': trace.subject,
            'split': split_by_index[idx],
        }
        for idx, trace in enumerate(fact_results)
    ])

    def traces_to_matrices(traces):
        centers = np.arange(num_layers)
        ie = np.zeros((len(traces), num_layers), dtype=np.float32)
        norm = np.zeros((len(traces), num_layers), dtype=np.float32)
        restore = np.zeros((len(traces), num_layers, NUM_NOISE_SAMPLES), dtype=np.float32)
        corrupt = np.zeros((len(traces), NUM_NOISE_SAMPLES), dtype=np.float32)
        for fact_i, trace in enumerate(traces):
            corrupt[fact_i] = trace.corrupt_probabilities
            for window in trace.windows:
                ie[fact_i, window.center] = window.mean_ie
                norm[fact_i, window.center] = window.normalized_recovery
                restore[fact_i, window.center, :] = window.restore_probabilities
        return centers, ie, norm, restore, corrupt

    def aggregate_stats(ie_matrix):
        return {
            'mean_ie': ie_matrix.mean(axis=0),
            'std_ie': ie_matrix.std(axis=0),
            'sem_ie': ie_matrix.std(axis=0) / max(np.sqrt(ie_matrix.shape[0]), 1.0),
        }

    centers, discovery_ie, discovery_norm, discovery_restore_probs, discovery_corrupt_probs = traces_to_matrices(discovery_traces)
    discovery_stats = aggregate_stats(discovery_ie)
    discovery_ci_low, discovery_ci_high = bootstrap_ci(discovery_ie, seed=SEED + 100)
    discovery_windows = pd.DataFrame([
        {
            **window_metadata(int(center)),
            'num_facts': len(discovery_traces),
            'discovery_mean_ie': float(discovery_stats['mean_ie'][center]),
            'discovery_std_ie': float(discovery_stats['std_ie'][center]),
            'discovery_sem_ie': float(discovery_stats['sem_ie'][center]),
            'discovery_ci_lower': float(discovery_ci_low[center]),
            'discovery_ci_upper': float(discovery_ci_high[center]),
        }
        for center in centers
    ])
    eligible_discovery = discovery_windows[discovery_windows['window_is_full_width']].copy()
    if eligible_discovery.empty:
        raise RuntimeError(f'{model_config}: no full-width windows; reduce WINDOW_SIZE')
    discovery_row = eligible_discovery.sort_values(
        ['discovery_mean_ie', 'window_center'],
        ascending=[False, True],
    ).iloc[0]
    discovery_center = int(discovery_row['window_center'])
    discovery_window = window_metadata(discovery_center)

    centers, confirmation_ie, confirmation_norm, confirmation_restore_probs, confirmation_corrupt_probs = traces_to_matrices(confirmation_traces)
    confirmation_stats = aggregate_stats(confirmation_ie)
    confirmation_ci_low, confirmation_ci_high = bootstrap_ci(confirmation_ie, seed=SEED + 200)
    confirmation_windows = pd.DataFrame([
        {
            **window_metadata(int(center)),
            'num_facts': len(confirmation_traces),
            'confirmation_mean_ie': float(confirmation_stats['mean_ie'][center]),
            'confirmation_std_ie': float(confirmation_stats['std_ie'][center]),
            'confirmation_sem_ie': float(confirmation_stats['sem_ie'][center]),
            'confirmation_ci_lower': float(confirmation_ci_low[center]),
            'confirmation_ci_upper': float(confirmation_ci_high[center]),
            'preselected_for_confirmation': bool(int(center) == discovery_center),
        }
        for center in centers
    ])
    confirmation_row = confirmation_windows[
        confirmation_windows['window_center'] == discovery_center
    ].iloc[0]
    enough_confirmation_facts = len(confirmation_traces) >= int(MIN_CONFIRMATION_FACTS)
    confirmation_passed = bool(
        enough_confirmation_facts
        and np.isfinite(float(confirmation_row['confirmation_ci_lower']))
        and float(confirmation_row['confirmation_ci_lower']) > 0
    )
    if not enough_confirmation_facts:
        selection_failure_reason = 'insufficient_confirmation_facts'
    elif not confirmation_passed:
        selection_failure_reason = 'confirmation_ci_not_positive'
    else:
        selection_failure_reason = None

    selected_trace_center = discovery_center if confirmation_passed else None
    selected_window_layers = [int(layer) for layer in discovery_window['window_layers']]
    selection_diagnostics = {
        'selection_method': 'discovery_argmax_then_held_out_confirmation',
        'eligible_window_rule': 'full_width_only',
        'tie_break_rule': 'lower_center',
        'minimum_confirmation_facts': int(MIN_CONFIRMATION_FACTS),
        'confirmation_passed': confirmation_passed,
        'selection_failure_reason': selection_failure_reason,
        'discovery_mean_ie': float(discovery_row['discovery_mean_ie']),
        'confirmation_mean_ie': float(confirmation_row['confirmation_mean_ie']),
        'confirmation_ci_lower': float(confirmation_row['confirmation_ci_lower']),
        'confirmation_ci_upper': float(confirmation_row['confirmation_ci_upper']),
    }

    trace_plot_path = out_dir / 'early_site_trace.png'
    final_selection = {
        'model_config': model_config,
        'model_name': model_name,
        'trace_plot_path': str(trace_plot_path),
        'selection_method': 'discovery_argmax_then_held_out_confirmation',
        'held_out_confirmed_window': confirmation_passed,
        'selection_failure_reason': selection_failure_reason,
        'strict_selection_failure_reason': selection_failure_reason,
        'selected_trace_center': selected_trace_center,
        'discovery_trace_center': discovery_center,
        'trace_window_start': int(discovery_window['window_start']),
        'trace_window_end': int(discovery_window['window_end']),
        'trace_window_layers': selected_window_layers,
        'config_layer_used_for_selection': False,
        'config_reference_layer': None if config_reference_layer is None else int(config_reference_layer),
        'config_reference_layer_is_graph_only': True,
        'adapter_validate_all_layers': ADAPTER_VALIDATE_ALL_LAYERS,
        'trust_remote_code_used': model_trust_remote_code,
        'selection_diagnostics': selection_diagnostics,
        'noise_calibration': noise_calibration,
        'num_fact_results': len(fact_results),
        'num_discovery_facts': len(discovery_traces),
        'num_confirmation_facts': len(confirmation_traces),
    }
    print(json.dumps(final_selection, indent=2))
    if not confirmation_passed:
        print(
            f'No held-out-confirmed window selected: {selection_failure_reason}. '
            f'Discovery center={discovery_center}.',
            flush=True,
        )

    full_mask = confirmation_windows['window_is_full_width'].to_numpy(dtype=bool)
    partial_mask = ~full_mask
    fig, ax = plt.subplots(figsize=(15.5, 6.5))
    ax.bar(
        centers[partial_mask],
        confirmation_stats['mean_ie'][partial_mask],
        color='lightgray',
        label='partial boundary windows',
    )
    ax.bar(
        centers[full_mask],
        confirmation_stats['mean_ie'][full_mask],
        color='steelblue',
        alpha=0.55,
        label='confirmation mean IE',
    )
    ax.fill_between(
        centers,
        confirmation_ci_low,
        confirmation_ci_high,
        color='tab:blue',
        alpha=0.15,
        label=f'{int(CONFIDENCE_LEVEL * 100)}% confirmation CI',
    )
    ax.plot(
        centers,
        discovery_stats['mean_ie'],
        color='black',
        linewidth=1.8,
        linestyle='--',
        label='discovery mean IE',
    )
    ax.axhline(0.0, color='black', linewidth=0.8)
    selection_color = 'tab:green' if confirmation_passed else 'darkorange'
    selection_label = 'held-out confirmed' if confirmation_passed else 'not held-out confirmed'
    ax.axvline(
        discovery_center,
        color=selection_color,
        linewidth=2.8,
        label=f'discovery center {discovery_center} ({selection_label})',
    )
    if config_reference_layer is not None:
        ax.axvline(
            int(config_reference_layer),
            color='tab:purple',
            linestyle='-.',
            linewidth=2.5,
            label=f'config reference layer {config_reference_layer} (graph only)',
        )
    ax.set_title(f'Latium causal tracing: {model_config}')
    ax.set_xlabel('MLP window center')
    ax.set_ylabel('Mean paired indirect effect across facts')
    ax.legend(fontsize=8, loc='upper left')
    plt.tight_layout()
    if SAVE:
        out_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(trace_plot_path, dpi=180)
    plt.show()

    if SAVE:
        out_dir.mkdir(parents=True, exist_ok=True)
        summary.to_csv(out_dir / 'summary_facts.csv', index=False)
        rejections_df.to_csv(out_dir / 'rejections.csv', index=False)
        split_assignments.to_csv(out_dir / 'split_assignments.csv', index=False)
        discovery_windows.to_csv(out_dir / 'discovery_windows.csv', index=False)
        confirmation_windows.to_csv(out_dir / 'confirmation_windows.csv', index=False)
        (out_dir / 'final_selection.json').write_text(json.dumps(json_safe(final_selection), indent=2))
        (out_dir / 'selection_diagnostics.json').write_text(json.dumps(json_safe(selection_diagnostics), indent=2))
        (out_dir / 'config.json').write_text(json.dumps(json_safe({
            'model_config': model_config,
            'model_preset': preset,
            'trust_remote_code_used': model_trust_remote_code,
            'config_reference_layer': config_reference_layer,
            'config_reference_layer_is_graph_only': True,
            'adapter_validate_all_layers': ADAPTER_VALIDATE_ALL_LAYERS,
            'num_valid_facts_requested': target_valid_facts,
            'num_valid_facts_global_default': NUM_VALID_FACTS,
            'num_noise_samples': NUM_NOISE_SAMPLES,
            'noise_calibration': noise_calibration,
            'noise_batch_size': NOISE_BATCH_SIZE,
            'window_mode': WINDOW_MODE,
            'window_size': primary_window_size,
            'discovery_fraction': DISCOVERY_FRACTION,
            'minimum_confirmation_facts': MIN_CONFIRMATION_FACTS,
            'bootstrap_samples': BOOTSTRAP_SAMPLES,
            'confidence_level': CONFIDENCE_LEVEL,
            'seed': SEED,
            'counts': counts,
            'mlp_output_modules': module_map.to_dict(orient='records'),
        }), indent=2))
        with (out_dir / 'fact_results.jsonl').open('w') as f:
            for trace in fact_results:
                f.write(json.dumps(json_safe({
                    'prompt_idx': trace.prompt_idx,
                    'prompt': trace.prompt,
                    'subject': trace.subject,
                    'target_full_text': trace.target_full_text,
                    'target_first_token_id': trace.target_first_token_id,
                    'target_first_token_text': trace.target_first_token_text,
                    'target_num_tokens': trace.target_num_tokens,
                    'clean_probability': trace.clean_probability,
                    'mean_corrupt_probability': trace.mean_corrupt_probability,
                    'total_effect': trace.total_effect,
                    'corrupt_relative_std': trace.corrupt_relative_std,
                    'windows': [
                        {
                            'window_center': w.center,
                            'window_start': w.start,
                            'window_end': w.end,
                            'window_size_actual': len(w.layers),
                            'window_layers': w.layers,
                            'window_is_full_width': w.is_full_width,
                            'mean_ie': w.mean_ie,
                            'median_ie_diagnostic': w.median_ie,
                            'mean_normalized_recovery_diagnostic': w.normalized_recovery,
                        }
                        for w in trace.windows
                    ],
                })) + '\n')
        np.savez_compressed(
            out_dir / 'raw_window_probabilities.npz',
            discovery_ie=discovery_ie,
            confirmation_ie=confirmation_ie,
            discovery_restore_probabilities=discovery_restore_probs,
            confirmation_restore_probabilities=confirmation_restore_probs,
            discovery_corrupt_probabilities=discovery_corrupt_probs,
            confirmation_corrupt_probabilities=confirmation_corrupt_probs,
            discovery_norm_diagnostic=discovery_norm,
            confirmation_norm_diagnostic=confirmation_norm,
            discovery_ci_lower=discovery_ci_low,
            discovery_ci_upper=discovery_ci_high,
            confirmation_ci_lower=confirmation_ci_low,
            confirmation_ci_upper=confirmation_ci_high,
        )
        print(f'Wrote outputs to {out_dir}')

    result = {**final_selection, **counts, 'out_dir': str(out_dir)}
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


## 8. Run Causal Tracing

This is the only execution stage. For each model it calibrates noise on candidate subject-token embeddings, collects valid paired traces, fixes a discovery window, tests only that window on held-out facts, and writes auditable CSV/JSON/NPZ plus the trace graph.


In [ ]:
batch_results = []
trace_by_model = {}

if RUN_CAUSAL_TRACE:
    for model_config in MODEL_CONFIGS:
        print(f'\n##### CAUSAL TRACE MODEL: {model_config} #####', flush=True)
        trace = run_one_model(model_config, dataset)
        batch_results.append(trace)
        trace_by_model[model_config] = trace
else:
    print('RUN_CAUSAL_TRACE=False; skipping causal tracing.')

batch_summary = pd.DataFrame(batch_results)
if len(batch_summary):
    display(batch_summary)

if SAVE:
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    batch_summary_path = OUT_ROOT / f'causal_trace_batch_summary_{RUN_TIMESTAMP}.csv'
    batch_summary.to_csv(batch_summary_path, index=False)
    print(f'Wrote causal trace batch summary to {batch_summary_path}')
